In [118]:
import numpy as np
import pandas as pd
import plotly.express as px
import os
import plotly.graph_objects as go
import copy


In [119]:
# get survex data
surveyDirectory = 'C:\\data\\udacity\\stackoverflow_survey'
rawData = {}
for folder in os.scandir(surveyDirectory):
    if folder.is_dir() and not str(folder.name) == "Archiv":
        surveyResults = pd.read_csv(surveyDirectory + "\\" + str(folder.name) + '\\survey_results_public.csv')
        rawData[str(folder.name)] = surveyResults

#surveySelection = copy.deepcopy(rawData)

C:\Users\sce2rng\AppData\Local\Temp\ipykernel_26936\2367963223.py:6: DtypeWarning:

Columns (8,12,13,14,15,16,50,51,52,53,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128) have mixed types. Specify dtype option on import or set low_memory=False.



In [120]:
#get survey question and tags
rawSchemas = {}
for folder in os.scandir(surveyDirectory):
    if folder.is_dir() and not str(folder.name) == "Archiv":
        if str(folder.name) <= "2020":
            surveySchema = pd.read_csv(surveyDirectory + "\\" + str(folder.name) + '\\survey_results_schema.csv', index_col=0)

        else:   # get only the short tag and teh question , in the newer survey schemas are also a QID and some explanations
            surveySchema = pd.read_csv(surveyDirectory + "\\" + str(folder.name) + '\\survey_results_schema.csv')
            surveySchema = surveySchema[surveySchema['qid'].str.contains('QID')]
            surveySchema = surveySchema[['qname', 'question']].set_index('qname')
            surveySchema = surveySchema.dropna()
        rawSchemas[str(folder.name)] = surveySchema.squeeze().to_dict()
surveySchemas = copy.deepcopy(rawSchemas)

In [121]:
os.scandir(surveyDirectory)

In [122]:
# define functions
def getQuestionText(questionId, surveySchemas):
    """
    function to get question to the question tags
    """
    idExistPerYear = {}
    for schema in surveySchemas.keys(): # look in each year
        print(schema)
        exist = False
        if questionId in surveySchemas[schema]:
            print(surveySchemas[schema][questionId])
            exist = True
        else:
            print(f"{questionId} not found in {schema}")
        idExistPerYear[schema] = exist
    series = pd.Series(idExistPerYear, name=questionId)
    return series

In [123]:
surveySelection = {}
columnsShort = ["Gender", "MainBranch", "Professional", "EdLevel", "FormalEducation", "Salary", "ConvertedSalary", "EmploymentStatus", 'Employment', "SalaryType", "YearsCode", "YearsCodePro", "CompTotal", "CompFreq", "ConvertedComp", "LanguageWorkedWith", "LanguageDesireNextYear", "HaveWorkedLanguage", "WantWorkLanguage", "Language"]

# Filter out columns that are not present in the dataframe
for year, df in rawData.items():
    columnsPresent = list(df.columns)
    columnsRemove = [col for col in columnsPresent if col not in columnsShort]
    surveySelection[year] = df.drop(columns=columnsRemove)

## Allign Coding experience

In [125]:
neededQuestions = ["LearnCode", 'YearsCode', "YearsCoding", 'YearsCodePro', "YearsCodingProf"]
idExistPerYear = pd.DataFrame()
for qId in neededQuestions:
    print("")
    print(qId)
    
    idExistPerYearSeries = getQuestionText(qId, rawSchemas)
    idExistPerYear = pd.concat([idExistPerYear, idExistPerYearSeries], axis=1)
display(idExistPerYear)


LearnCode
2017
LearnCode not found in 2017
2018
LearnCode not found in 2018
2019
LearnCode not found in 2019
2020
LearnCode not found in 2020
2021
How did you learn to code? Select all that apply.
2022
How did you learn to code? Select all that apply.
2023
How do you learn to code? Select all that apply.

YearsCode
2017
YearsCode not found in 2017
2018
YearsCode not found in 2018
2019
Including any education, how many years have you been coding?
2020
Including any education, how many years have you been coding in total?
2021
Including any education, how many years have you been coding in total?
2022
Including any education, how many years have you been coding in total?
2023
Including any education, how many years have you been coding in total?

YearsCoding
2017
YearsCoding not found in 2017
2018
Including any education, for how many years have you been coding?
2019
YearsCoding not found in 2019
2020
YearsCoding not found in 2020
2021
YearsCoding not found in 2021
2022
YearsCoding not 

,LearnCode,YearsCode,YearsCoding,YearsCodePro,YearsCodingProf,LanguageWorkedWith,LanguageDesireNextYear
2017,False,False,False,False,False,False,False
2018,False,False,True,False,True,True,True
2019,False,True,False,True,False,True,True
2020,False,True,False,True,False,True,True
2021,True,True,False,True,False,False,False
2022,True,True,False,True,False,False,False
2023,True,True,False,True,False,False,False


In [126]:
# yearsCodePro = YearsCodingProf and YearsCode = YearsCoding
surveySelection["2018"] = surveySelection["2018"].rename(columns={"YearsCoding":'YearsCode'})
surveySelection["2018"] = surveySelection["2018"].rename(columns={"YearsCodingProf":'YearsCodePro'})

## Allign annual compensation

In [127]:
neededQuestions = [ "Salary", "SalaryType", "ConvertedSalary", 'CompTotal', 'CompFreq', 'ConvertedComp']# CodingActivities,"UndergradMajor"
idExistPerYear = pd.DataFrame()
for qId in neededQuestions:
    print("")
    print(qId)
    
    idExistPerYearSeries = getQuestionText(qId, rawSchemas)
    idExistPerYear = pd.concat([idExistPerYear, idExistPerYearSeries], axis=1)
display(idExistPerYear)


Salary
2017
What is your current annual base salary, before taxes, and excluding bonuses, grants, or other compensation?
2018
What is your current gross salary (before taxes and deductions), in ${q://QID50/ChoiceGroup/SelectedChoicesTextEntry}? Please enter a whole number in the box below, without any punctuation. If you are paid hourly, please estimate an equivalent weekly, monthly, or yearly salary. If you prefer not to answer, please leave the box empty.
2019
Salary not found in 2019
2020
Salary not found in 2020
2021
Salary not found in 2021
2022
Salary not found in 2022
2023
Salary not found in 2023

SalaryType
2017
SalaryType not found in 2017
2018
Is that salary weekly, monthly, or yearly?
2019
SalaryType not found in 2019
2020
SalaryType not found in 2020
2021
SalaryType not found in 2021
2022
SalaryType not found in 2022
2023
SalaryType not found in 2023

ConvertedSalary
2017
ConvertedSalary not found in 2017
2018
Salary converted to annual USD salaries using the exchange rat

,Salary,SalaryType,ConvertedSalary,CompTotal,CompFreq,ConvertedComp
2017,True,False,False,False,False,False
2018,True,True,True,False,False,False
2019,False,False,False,True,True,True
2020,False,False,False,True,True,True
2021,False,False,False,True,True,False
2022,False,False,False,True,True,False
2023,False,False,False,True,False,False


In [128]:
# get compensation per year for each datasaet
for year in ["2019","2020"]:
    surveySelection[year] = surveySelection[year].rename(columns={"ConvertedComp":'AnnualComp'})
    surveySchemas[year]['AnnualComp'] = surveySchemas[year]["ConvertedComp"]
    del surveySchemas[year]['ConvertedComp']

surveySelection["2018"] = surveySelection["2018"].rename(columns={"ConvertedSalary":'AnnualComp'})
surveySchemas["2018"]['AnnualComp'] = surveySchemas["2018"]["ConvertedSalary"]
del surveySchemas["2018"]['ConvertedSalary']

# keep in mind: 2017 salary is without bonuses and benefits
surveySelection["2017"] = surveySelection["2017"].rename(columns={"Salary":'AnnualComp'})
surveySchemas["2017"]['AnnualComp'] = surveySchemas["2017"]["Salary"]
del surveySchemas["2017"]['Salary']

surveySelection["2023"] = surveySelection["2023"].rename(columns={"CompTotal":'AnnualComp'})
surveySchemas["2023"]['AnnualComp'] = surveySchemas["2023"]["CompTotal"]
del surveySchemas["2023"]['CompTotal']


In [129]:
# calculate annual compensation for 2021 and 2022
def get_year_factor(freq):
    if freq == 'Weekly':
        return 50
    elif freq == 'Monthly':
        return 12
    elif freq == 'Yearly':
        return 1
    else:
        return 0
    
for year in ["2021","2022"]:
    surveySelection[year]["CompFreq"] = surveySelection[year]["CompFreq"].apply(get_year_factor)
    surveySelection[year]['CompFreq'] = surveySelection[year].apply(lambda row: 1 if row['CompTotal'] == 0 else row['CompFreq'], axis=1)
    surveySelection[year]['AnnualComp'] = surveySelection[year]["CompTotal"] * surveySelection[year]['CompFreq']

for year, df in surveySelection.items():  
    df['AnnualComp'] = df['AnnualComp'].astype('float64')


## Allign Employment and Education

In [130]:
neededQuestions = ["EdLevel", "FormalEducation", "Employment","EmploymentStatus","EducationTypes"]# CodingActivities,"UndergradMajor"
idExistPerYear = pd.DataFrame()
for qId in neededQuestions:
    print("")
    print(qId)
    
    idExistPerYearSeries = getQuestionText(qId, rawSchemas)
    idExistPerYear = pd.concat([idExistPerYear, idExistPerYearSeries], axis=1)
display(idExistPerYear)


EdLevel
2017
EdLevel not found in 2017
2018
EdLevel not found in 2018
2019
Which of the following best describes the highest level of formal education that you’ve completed?
2020
Which of the following best describes the highest level of formal education that you’ve completed?
2021
Which of the following best describes the highest level of formal education that you’ve completed? *
2022
Which of the following best describes the highest level of formal education that you’ve completed? *
2023
Which of the following best describes the highest level of formal education that you’ve completed? *

FormalEducation
2017
Which of the following best describes the highest level of formal education that you've completed?
2018
Which of the following best describes the highest level of formal education that you’ve completed?
2019
FormalEducation not found in 2019
2020
FormalEducation not found in 2020
2021
FormalEducation not found in 2021
2022
FormalEducation not found in 2022
2023
FormalEducation n

,EdLevel,FormalEducation,Employment,EmploymentStatus,EducationTypes
2017,False,True,False,True,True
2018,False,True,True,False,True
2019,True,False,True,False,False
2020,True,False,True,False,False
2021,True,False,True,False,False
2022,True,False,True,False,False
2023,True,False,True,False,False


In [131]:
# FormalEducation is the same as EdLevel in teh newer surveys
for year in ["2017","2018"]:
    surveySelection[year] = surveySelection[year].rename(columns={"FormalEducation":'EdLevel'})
    surveySchemas[year]['EdLevel'] = surveySchemas[year]["FormalEducation"]
    del surveySchemas[year]['FormalEducation']

# EmploymentStatus is the same as Employment in the newer surveys
for year in ["2017"]:
    surveySelection[year] = surveySelection[year].rename(columns={"EmploymentStatus":'Employment'})
    surveySchemas[year]['Employment'] = surveySchemas[year]["EmploymentStatus"]
    del surveySchemas[year]['EmploymentStatus']
    



In [132]:
surveySelection["2017"].columns

Index(['Professional', 'Employment', 'EdLevel', 'Gender', 'AnnualComp'], dtype='object')

In [133]:
surveySelection["2022"]["Employment"].value_counts()

Employed, full-time                                                                                                                    42962
Student, full-time                                                                                                                      6756
Independent contractor, freelancer, or self-employed                                                                                    4978
Employed, full-time;Independent contractor, freelancer, or self-employed                                                                3486
Not employed, but looking for work                                                                                                      1831
                                                                                                                                       ...  
Student, part-time;Independent contractor, freelancer, or self-employed;Retired                                                            1
Employed, ful

In [134]:
# handel not employed
def align_employment(employment):
    if employment == "Employed full-time" or employment == "Employed part-time" or employment == "Independent contractor, freelancer, or self-employed":
        return "Employed"
    elif employment == "Employed, full-time" or employment == "Employed, part-time":
        return "Employed"
    elif employment == "Not employed, and not looking for work" or employment == "Not employed, but looking for work":
        return "Unemployed"
    elif employment == "Retired":
        return "Retired"
    elif employment == "Student" or employment == "Student, full-time" or employment == "Student, part-time":
        return "Student"
    elif employment == "I prefer not to say":
        return None
    elif pd.isnull(employment):
        return None
    elif employment.count("full-time") > 1 or ("Not employed," in employment and "Employed," in employment):
        return None
    elif "Employed," in employment:
        return "Employed"
    else:
        return None

for year in surveySelection.keys():
    surveySelection[year]["EmpUnemp"] = surveySelection[year]["Employment"].apply(align_employment)
    if "AnnualComp" in surveySelection[year].columns:
        print(year)
        count_zeros = ((surveySelection[year]['AnnualComp'] == 0) & (surveySelection[year]['EmpUnemp'] == 'Employed')).sum()
        print(f"zero annual compensation but Employed: {count_zeros}")
surveySelection[year]["EmpUnemp"].value_counts()


2017


zero annual compensation but Employed: 6
2018
zero annual compensation but Employed: 207
2019
zero annual compensation but Employed: 194
2020
zero annual compensation but Employed: 137
2021
zero annual compensation but Employed: 257
2022
zero annual compensation but Employed: 138
2023
zero annual compensation but Employed: 111


Employed      71271
Student        8181
Unemployed     3397
Retired         570
Name: EmpUnemp, dtype: int64

## Allign Gender and Main Branch

In [135]:
neededQuestions = ["Gender", "MainBranch", "Professional"]
idExistPerYear = pd.DataFrame()
for qId in neededQuestions:
    print("")
    print(qId)
    
    idExistPerYearSeries = getQuestionText(qId, rawSchemas)
    idExistPerYear = pd.concat([idExistPerYear, idExistPerYearSeries], axis=1)
display(idExistPerYear)


Gender
2017
Which of the following do you currently identify as?
2018
Which of the following do you currently identify as? Please select all that apply. If you prefer not to answer, you may leave this question blank.
2019
Which of the following do you currently identify as? Please select all that apply. If you prefer not to answer, you may leave this question blank.
2020
Which of the following describe you, if any? Please check all that apply. If you prefer not to answer, you may leave this question blank.
2021
Which of the following describe you, if any? Please check all that apply.
2022
Which of the following describe you, if any? Please check all that apply.
2023
Gender not found in 2023

MainBranch
2017
MainBranch not found in 2017
2018
MainBranch not found in 2018
2019
Which of the following options best describes you today? Here, by "developer" we mean "someone who writes code."
2020
Which of the following options best describes you today? Here, by "developer" we mean "someone w

,Gender,MainBranch,Professional
2017,True,False,True
2018,True,False,False
2019,True,True,False
2020,True,True,False
2021,True,True,False
2022,True,True,False
2023,False,True,False


In [136]:
# Professional is the same as MainBranch in the newer surveys
surveySelection["2017"] = surveySelection["2017"].rename(columns={"Professional":'MainBranch'})
surveySchemas["2017"]['MainBranch'] = surveySchemas["2017"]["Professional"]
del surveySchemas["2017"]['Professional']

In [137]:
for year, df in surveySelection.items():
    if "MainBranch" in df.columns:
        print(year)
        print(df["MainBranch"].value_counts())

2017
Professional developer                                  36131
Student                                                  8224
Professional non-developer who sometimes writes code     5140
Used to be a professional developer                       983
None of these                                             914
Name: MainBranch, dtype: int64
2019
I am a developer by profession                                                   65679
I am a student who is learning to code                                           10189
I am not primarily a developer, but I write code sometimes as part of my work     7539
I code primarily as a hobby                                                       3340
I used to be a developer by profession, but no longer am                          1584
Name: MainBranch, dtype: int64
2020
I am a developer by profession                                                   47193
I am a student who is learning to code                                            7970
I am

In [138]:
def align_mainbranch_2017(employment):
    if employment == "Professional developer":
        return "I am a developer by profession"
    elif employment == "Professional non-developer who sometimes writes code":
        return "I am not primarily a developer, but I write code sometimes as part of my work"
    elif employment == "Student":
        return "I am a student who is learning to code"
    elif employment == "Used to be a professional developer":
        return "I used to be a developer by profession, but no longer am"

surveySelection["2017"]["MainBranch"] = surveySelection["2017"]["MainBranch"].apply(align_mainbranch_2017)

In [140]:
for year, df in surveySelection.items():
    if "Gender" in df.columns:
        print(year)
        print(df["Gender"].value_counts())
        nanCount = df["Gender"].isna().sum()
        print(f"nans: {nanCount}")

2017
Male                                                       31589
Female                                                      2600
Other                                                        225
Male; Other                                                  171
Gender non-conforming                                        160
Male; Gender non-conforming                                   65
Female; Transgender                                           56
Transgender                                                   55
Female; Gender non-conforming                                 29
Male; Female                                                  15
Transgender; Gender non-conforming                            15
Male; Female; Transgender; Gender non-conforming; Other       15
Male; Transgender                                             11
Female; Transgender; Gender non-conforming                     8
Male; Female; Transgender; Gender non-conforming               7
Male; Female; Transg

In [141]:

def align_gender(gender):
    if gender == "Man":
        return "Male"
    elif gender == "Woman":
        return "Female"
    elif gender == "Male":
        return "Male"
    elif gender == "Female":
        return "Female"
    elif gender == "Prefer not to say":
        return None
    elif gender == "I prefer not to answer":
        return None
    elif pd.isnull(gender):
        return None
    else:
        return "Other"
    
for year,df in surveySelection.items():
    if "Gender" in df.columns:
        df["GenderCat"] = df["Gender"].apply(align_gender)

print(surveySelection["2021"]["GenderCat"].value_counts())
nanCount = surveySelection["2021"]["GenderCat"].isna().sum()
print(f"nans: {nanCount}")


Male      74817
Female     4120
Other      1907
Name: GenderCat, dtype: int64
nans: 2595


## Allign Coding Languages

In [150]:
neededQuestions = ["LanguageWorkedWith", "LanguageDesireNextYear", "HaveWorkedLanguage", "WantWorkLanguage", "Language"]
idExistPerYear = pd.DataFrame()
for qId in neededQuestions:
    print("")
    print(qId)
    
    idExistPerYearSeries = getQuestionText(qId, rawSchemas)
    idExistPerYear = pd.concat([idExistPerYear, idExistPerYearSeries], axis=1)
display(idExistPerYear)


LanguageWorkedWith
2017
LanguageWorkedWith not found in 2017
2018
Which of the following programming, scripting, and markup languages have you done extensive development work in over the past year, and which do you want to work in over the next year?  (If you both worked with the language and want to continue to do so, please check both boxes in that row.)
2019
Which of the following programming, scripting, and markup languages have you done extensive development work in over the past year, and which do you want to work in over the next year?  (If you both worked with the language and want to continue to do so, please check both boxes in that row.)
2020
Which programming, scripting, and markup languages have you done extensive development work in over the past year, and which do you want to work in over the next year? (If you both worked with the language and want to continue to do so, please check both boxes in that row.)
2021
LanguageWorkedWith not found in 2021
2022
LanguageWorkedW

,LanguageWorkedWith,LanguageDesireNextYear,HaveWorkedLanguage,WantWorkLanguage,Language
2017,False,False,True,True,False
2018,True,True,False,False,False
2019,True,True,False,False,False
2020,True,True,False,False,False
2021,False,False,False,False,True
2022,False,False,False,False,True
2023,False,False,False,False,True


In [149]:
for year, schema in surveySchemas.items():
    print(year)
    for tag, question in schema.items():
        if "language" in question.lower():
            
            print(f"{tag}: {question}")


2017
AssessJobTech: When you're assessing potential jobs to apply to, how important are each of the following to you? The languages, frameworks, and other technologies I'd be working with
HaveWorkedLanguage: Which of the following languages have you done extensive development work in over the past year, and which do you want to work in over the next year?
WantWorkLanguage: Which of the following languages have you done extensive development work in over the past year, and which do you want to work in over the next year?
2018
AssessJob4: Imagine that you are assessing a potential job opportunity. Please rank the following aspects of the job opportunity in order of importance (by dragging the choices up and down), where 1 is the most important and 10 is the least important. The languages, frameworks, and other technologies I'd be working with
LanguageWorkedWith: Which of the following programming, scripting, and markup languages have you done extensive development work in over the past y

In [142]:
# Get median values for annual compensation per year and gender
onlyEmp = {}
for year, df in surveySelection.items():
    onlyEmp[year] = df[(df['EmpUnemp'] == "Employed") & (df['AnnualComp'] != 0)]



## Gender pay gap

In [143]:
# make one dataframe with with year, gender and annual compensation
data = []
for year, df in onlyEmp.items():
    if "AnnualComp" in df.columns and "GenderCat" in df.columns:
        compSeries = df.groupby('GenderCat')["AnnualComp"].median()
        # get annual compensation for each year and gender-group
        for gender, comp in compSeries.items():
            data.append({'Year': year, 'Gender': gender, 'AnnualComp': comp})

df = pd.DataFrame(data)

# create plot with annual compensation over year with gender in different colors
fig = go.Figure()
fig = px.line(df, x='Year', y='AnnualComp', color='Gender',
                 title='Annual Compensation by Gender Over Years',
                 labels={'AnnualComp': 'Annual Compensation Median', 'Gender': 'Gender'})

# Set the size of the figure
fig.update_layout(width=600, height=400)

# Show the plot
fig.show()

## Main Branch


In [144]:

for year, df in onlyEmp.items():
    if "AnnualComp" in df.columns and "MainBranch" in df.columns:
        compSeries = df.groupby('MainBranch')["AnnualComp"].median()
        compSeries = compSeries.dropna()
        # get annual compensation for each year and gender-group
        for mainBranch, comp in compSeries.items():
            data.append({'Year': year, 'MainBranch': mainBranch, 'AnnualComp': comp})

df = pd.DataFrame(data)

# create plot with annual compensation 
fig = go.Figure()
fig = px.line(df, x='Year', y='AnnualComp', color='MainBranch',
                title='Annual Compensation by Main Branch',
                labels={'AnnualComp': 'Annual Compensation Median', 'MainBranch': 'Main Branch'},
                markers=True)

# Set the size of the figure
fig.update_layout(width=800, height=400)

# Show the plot
fig.show()
del fig

## Language


In [ ]:
for year, df in onlyEmp.items():
    if "AnnualComp" in df.columns and "MainBranch" in df.columns:
        compSeries = df.groupby('MainBranch')["AnnualComp"].median()
        compSeries = compSeries.dropna()
        # get annual compensation for each year and gender-group
        for mainBranch, comp in compSeries.items():
            data.append({'Year': year, 'MainBranch': mainBranch, 'AnnualComp': comp})

df = pd.DataFrame(data)

# create plot with annual compensation 
fig = go.Figure()
fig = px.line(df, x='Year', y='AnnualComp', color='MainBranch',
                title='Annual Compensation by Main Branch',
                labels={'AnnualComp': 'Annual Compensation Median', 'MainBranch': 'Main Branch'},
                markers=True)

# Set the size of the figure
fig.update_layout(width=800, height=400)

# Show the plot
fig.show()
del fig